In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

paths = [
    "/content/drive/My Drive/Cybersecurity_DDoS/ddos_test_data.csv",
    "/content/drive/My Drive/Cybersecurity_DDoS/ddos_test_embeddings.csv",
    "/content/drive/My Drive/CIC_IDS(2017)_ZeroDay/multiclass_test_data.csv",
    "/content/drive/My Drive/CIC_IDS(2017)_ZeroDay/multiclass_CIC_IDS2017_test_embeddings.csv",
    "/content/drive/My Drive/malware_project/test_374Features.csv",
    "/content/drive/My Drive/malware_project/test_embeddings_374Features.csv"
]

for p in paths:
    print(p, "exists?" , os.path.exists(p))


/content/drive/My Drive/Cybersecurity_DDoS/ddos_test_data.csv exists? True
/content/drive/My Drive/Cybersecurity_DDoS/ddos_test_embeddings.csv exists? True
/content/drive/My Drive/CIC_IDS(2017)_ZeroDay/multiclass_test_data.csv exists? True
/content/drive/My Drive/CIC_IDS(2017)_ZeroDay/multiclass_CIC_IDS2017_test_embeddings.csv exists? True
/content/drive/My Drive/malware_project/test_374Features.csv exists? True
/content/drive/My Drive/malware_project/test_embeddings_374Features.csv exists? True


In [ ]:
import numpy as np
import pandas as pd

# ============================
# 1. Load TEST Original CSVs
# ============================

# DDoS
ddos_test = pd.read_csv("/content/drive/My Drive/Cybersecurity_DDoS/ddos_test_data.csv")
ddos_test_emb = pd.read_csv("/content/drive/My Drive/Cybersecurity_DDoS/ddos_test_embeddings.csv")

# Zero-Day
zero_test = pd.read_csv("/content/drive/My Drive/CIC_IDS(2017)_ZeroDay/multiclass_test_data.csv")
zero_test_emb = pd.read_csv("/content/drive/My Drive/CIC_IDS(2017)_ZeroDay/multiclass_CIC_IDS2017_test_embeddings.csv")

# Malware
mal_test = pd.read_csv("/content/drive/My Drive/malware_project/test_374Features.csv")
mal_test_emb = pd.read_csv("/content/drive/My Drive/malware_project/test_embeddings_374Features.csv")

print("Files Loaded Successfully!")


Files Loaded Successfully!


In [ ]:
# =============================
# DDoS LABELS
# =============================
ddos_labels = ddos_test["Label"].copy()

# =============================
# ZERO-DAY LABELS (shift 1–14 → 5–18)
# =============================
zero_labels = zero_test["Label"].copy()
zero_labels = zero_labels.apply(lambda x: x + 4 if x > 0 else 0)

# =============================
# MALWARE LABELS (corrected mapping)
# =============================
mal_map = {
    0: 19,   # Adware
    1: 20,   # Banking
    2: 21,   # SMS
    3: 22,   # Riskware
    4: 0     # Benign
}

mal_labels = mal_test["Class"].map(mal_map)


In [ ]:
print(sorted(set(ddos_labels)))
print(sorted(set(zero_labels)))
print(sorted(set(mal_labels)))

[0, 1, 2, 3, 4]
[0, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
[0, 19, 20, 21, 22]


In [ ]:
ddos_emb_arr = ddos_test_emb.values
zero_emb_arr = zero_test_emb.values
mal_emb_arr  = mal_test_emb.values

print(ddos_emb_arr.shape, zero_emb_arr.shape, mal_emb_arr.shape)


(71883, 128) (121316, 128) (2320, 128)


In [ ]:
# Creating Zero-padding for fusion (Each embedding is 128)

zero_block = np.zeros((1, 128))

# For DDoS samples
ddos_pad_zero1 = np.zeros((ddos_emb_arr.shape[0], 128))
ddos_pad_zero2 = np.zeros((ddos_emb_arr.shape[0], 128))

ddos_fused = np.concatenate([ddos_emb_arr,
                             ddos_pad_zero1,
                             ddos_pad_zero2], axis=1)

# For Zero-Day samples
zero_pad_zero1 = np.zeros((zero_emb_arr.shape[0], 128))
zero_pad_zero2 = np.zeros((zero_emb_arr.shape[0], 128))

zero_fused = np.concatenate([zero_pad_zero1,
                             zero_emb_arr,
                             zero_pad_zero2], axis=1)

# For Malware samples
mal_pad_zero1 = np.zeros((mal_emb_arr.shape[0], 128))
mal_pad_zero2 = np.zeros((mal_emb_arr.shape[0], 128))

mal_fused = np.concatenate([mal_pad_zero1,
                            mal_pad_zero2,
                            mal_emb_arr], axis=1)
print("Zero-padding applied. Embeddings fused")

Zero-padding applied. Embeddings fused


In [ ]:
# Combine all TEST samples into one dataset
X_test = np.vstack([ddos_fused, zero_fused, mal_fused])
y_test = np.concatenate([ddos_labels, zero_labels, mal_labels])

print("Final TEST Shape:", X_test.shape)
print("Final TEST Labels Shape:", y_test.shape)

Final TEST Shape: (195519, 384)
Final TEST Labels Shape: (195521,)


In [ ]:
print("ddos:", ddos_fused.shape[0], len(ddos_labels))
print("zero:", zero_fused.shape[0], len(zero_labels))
print("malware:", mal_fused.shape[0], len(mal_labels))

ddos: 71883 71884
zero: 121316 121317
malware: 2320 2320


In [ ]:
ddos_labels = ddos_labels[:ddos_fused.shape[0]]
zero_labels = zero_labels[:zero_fused.shape[0]]


In [ ]:
y_test = np.concatenate([ddos_labels, zero_labels, mal_labels])


In [ ]:
print("Final TEST Shape:", X_test.shape)
print("Final TEST Labels Shape:", y_test.shape)

Final TEST Shape: (195519, 384)
Final TEST Labels Shape: (195519,)


In [ ]:
# Save final concatenated TEST embeddings & labels as CSV

output_dir = "/content/drive/My Drive/Embedding_Concatenate"

# Convert to DataFrame
df_test = pd.DataFrame(X_test)
df_test["Label"] = y_test

# Save CSV
df_test.to_csv(f"{output_dir}/Concatenated_Test_embeddings.csv", index=False)

print(f"Saved successfully → {output_dir}/Concatenated_Test_embeddings.csv")


Saved successfully → /content/drive/My Drive/Embedding_Concatenate/Concatenated_Test_embeddings.csv
